# Wine-Intelligence
## Advanced Machine Learning — Final Project

---

### Project Summary

Wine-Intelligence is a sourcing intelligence tool for buyers at restaurants, hotels, and online retailers. Given a photograph of a wine label, the system automatically produces a structured sourcing brief covering wine style, quality tier, a full wine fact card, and taste-similar alternatives derived from a blind human tasting experiment.

| Component | Approach | Task |
| --- | --- | --- |
| CNN (Image model) | ResNet-18 (transfer learning) vs. custom CNN | Classify wine style: red / white / rosé / sparkling |
| BiLSTM (Text model) | Bidirectional LSTM vs. unidirectional LSTM | Classify quality tier: Entry / Premium / Exceptional |
| Taste similarity | Euclidean distance on napping coordinates | Surface 3 wines real humans rated as most similar in taste |

**Dataset:** WineSensed — Learning to Taste, NeurIPS 2023 · `https://huggingface.co/datasets/Dakhoo/L2T-NeurIPS-2023`  
**License:** CC BY-NC-ND 4.0 · Bender et al., NeurIPS 2023

---

### Development Phases

This project is developed in three phases. The notebook is designed to run correctly in all three without modification — only one variable changes between Phase 1 and Phase 2.

| Phase | Environment | Purpose |
| --- | --- | --- |
| **Phase 1** | VS Code (local, CPU) | Build the full pipeline end-to-end using the `small` dataset split. Goal: every cell runs without error. Accuracy is not the focus here. |
| **Phase 2** | Google Colab (GPU) | Upload this notebook to Colab. Set `DATASET_SPLIT = 'all'` in Section 2. Train for real, save weights to Google Drive, download back to local. |
| **Phase 3** | VS Code (local, CPU) | Load the saved weights. Complete explainability, business integration, deployment, and all written analysis. No GPU needed. |

> **How to switch to Phase 2:** In Section 2, change `DATASET_SPLIT = 'small'` to `DATASET_SPLIT = 'all'`. That is the only change required.

---

### Notebook Structure

| Section | Content |
| --- | --- |
| 1 | Environment setup — dependencies and imports |
| 2 | Data loading and sanity check |
| 3 | Exploratory data analysis — images |
| 4 | Exploratory data analysis — text reviews |
| 5 | Image preprocessing and data loaders |
| 6 | Text preprocessing and data loaders |
| 7 | CNN — custom architecture (trained from scratch) |
| 8 | CNN — ResNet-18 (transfer learning) |
| 9 | CNN explainability — Grad-CAM |
| 10 | LSTM — unidirectional baseline |
| 11 | BiLSTM — bidirectional with attention |
| 12 | BiLSTM explainability — attention weights |
| 13 | Taste similarity — napping Euclidean distances |
| 14 | Business integration — combined sourcing recommendation |
| 15 | Business framing, ethics, and team contributions |

---

## 1. Environment Setup

This section installs all required packages and imports the libraries used throughout the notebook.

> **Run once per environment.** On first use locally or on Google Colab, execute this section top to bottom before proceeding.
>
> **Colab note:** Remove the `--index-url` flag from the PyTorch install line — Colab provides a GPU-enabled build by default.

In [ ]:
# ── 1.1 Package installation ─────────────────────────────────────────────────
# Run once per environment. Inline comments are not allowed on %pip lines,
# so explanations are written above each install.
#
# CPU-only PyTorch for local development (Phase 1 and Phase 3).
# On Google Colab (Phase 2), remove '--index-url ...' to get the CUDA build.
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

# HuggingFace — loads the WineSensed dataset and provides tokenisers
%pip install datasets transformers

# Data handling, visualisation, and evaluation metrics
%pip install pandas numpy matplotlib seaborn scikit-learn

# Grad-CAM explainability for the CNN
%pip install torchcam

# Deployment prototype
%pip install streamlit

# Image handling and progress bars
%pip install Pillow tqdm

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ── 1.2 Imports ──────────────────────────────────────────────────────────────

import os
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Progress tracking
from tqdm import tqdm

# PyTorch — core deep learning framework used for both the CNN and BiLSTM
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# PyTorch — computer vision utilities and pre-trained models
import torchvision
import torchvision.transforms as transforms
from torchvision import models

# Scikit-learn — dataset splitting and evaluation metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ── Reproducibility ──────────────────────────────────────────────────────────
# A fixed random seed guarantees that every run produces identical train/val/test
# splits, weight initialisations, and data augmentation sequences.
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device selection ─────────────────────────────────────────────────────────
# PyTorch automatically uses the GPU when one is available (e.g., on Colab).
# Falls back to CPU for local development — all Phase 1 code runs on CPU.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Confirm all imports loaded successfully ───────────────────────────────────
print('Libraries loaded successfully:')
print(f'  numpy          {np.__version__}')
print(f'  pandas         {pd.__version__}')
print(f'  matplotlib     {plt.matplotlib.__version__}')
print(f'  seaborn        {sns.__version__}')
print(f'  Pillow         {Image.__version__}')
print(f'  torch          {torch.__version__}')
print(f'  torchvision    {torchvision.__version__}')
print(f'  scikit-learn   {__import__("sklearn").__version__}')
print()
print(f'Device  : {DEVICE}')
print(f'Seed    : {SEED}')


---

## 2. Data Loading and Sanity Check

The WineSensed dataset is hosted on HuggingFace and is split into four named configurations:

| Config | What it contains | When we use it |
| --- | --- | --- |
| `small` | A small representative sample | Phase 1 — local VS Code (CPU) |
| `all` | The full 897k images + 824k reviews | Phase 2 — Google Colab (GPU) |
| `vintages` | 350k vintage metadata records | Both phases |
| `napping_participants` | 5k+ pairwise taste distances from 256 tasters | Both phases |

We load three things in this section:
1. **Main split** (`small` or `all`) — contains `images_reviews_attributes.csv`: image links, reviews, and wine attributes (vintage_id, wine name, year, country, region, grape, alcohol, price, rating)
2. **Vintages** — structured wine metadata
3. **Napping** — the 2D taste-space coordinates used later for similarity search

> **To switch phases:** change `DATASET_SPLIT = 'small'` to `DATASET_SPLIT = 'all'` below. That is the only change needed.

In [ ]:
# ── 2.1 Phase control ────────────────────────────────────────────────────────
# This is the ONLY line that changes between Phase 1 and Phase 2.
# Phase 1 (VS Code, CPU)  → DATASET_SPLIT = 'small'
# Phase 2 (Colab, GPU)    → DATASET_SPLIT = 'all'

DATASET_SPLIT = 'small'

# ── 2.2 Load main dataset (images + reviews + attributes) ────────────────────
# images_reviews_attributes.csv fields used in this project:
#   vintage_id, image, review, experiment_id, year, wine, country,
#   region, grape, alcohol, price, rating

from datasets import load_dataset

print(f'Loading WineSensed [{DATASET_SPLIT}] split...')
ds_main = load_dataset('Dakhoo/L2T-NeurIPS-2023', DATASET_SPLIT)
print(f'  Main split loaded : {len(ds_main["train"]):,} records')
print(f'  Columns           : {ds_main["train"].column_names}')

# ── 2.3 Load vintage metadata ─────────────────────────────────────────────────
# Contains structured wine attributes: vintage_id, wine name, region, grape, etc.

print('\nLoading vintages...')
ds_vintages = load_dataset('Dakhoo/L2T-NeurIPS-2023', 'vintages')
print(f'  Vintages loaded   : {len(ds_vintages["train"]):,} records')
print(f'  Columns           : {ds_vintages["train"].column_names}')

# ── 2.4 Load napping (taste similarity coordinates) ───────────────────────────
# napping.csv fields: session_round_name, event_name, experiment_no,
#   experiment_id, coor1 (x), coor2 (y), color
# experiment_id links napping rows to wines in images_reviews_attributes.csv

print('\nLoading napping coordinates...')
ds_napping = load_dataset('Dakhoo/L2T-NeurIPS-2023', 'napping_participants')
print(f'  Napping loaded    : {len(ds_napping["train"]):,} records')
print(f'  Columns           : {ds_napping["train"].column_names}')

# ── 2.5 Convert to pandas DataFrames for easier handling ─────────────────────
df_main    = ds_main['train'].to_pandas()
df_vintage = ds_vintages['train'].to_pandas()
df_napping = ds_napping['train'].to_pandas()

print('\nSanity check — first row of main dataset:')
print(df_main.iloc[0])